# Importing the data

In [46]:
%pip install ucimlrepo
# Add this pip to the requirements folder which should also include Python version
from ucimlrepo import fetch_ucirepo 
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
  
# fetch dataset 
online_retail = fetch_ucirepo(id=352) 
  
# data (as pandas dataframes) 
data = online_retail.data.original 

Note: you may need to restart the kernel to use updated packages.


In [47]:
df = duckdb.sql(
'''
WITH RECURSIVE online_retail AS (
    SELECT
        b.Country                                                   AS country,
        b.Description                                               AS description,
        b.Quantity                                                  AS quantity,
        CAST(STRPTIME(b.InvoiceDate, '%m/%d/%Y %H:%M') AS DATE)     AS invoice_date,
        MIN(invoice_date) OVER(PARTITION BY CustomerID)             AS first_purchase_date,
        b.UnitPrice                                                 AS unit_price,
        CAST(b.CustomerID AS STRING)                                AS customer_id,
        b.InvoiceNo                                                 AS invoice_number,
        b.Quantity * b.UnitPrice                                    AS order_value
    FROM
        data b
),

snapshots AS (
    -- Anchor member - first date in dataset: 1/10/2010
    SELECT
        MIN(invoice_date) AS snapshot,
        MAX(invoice_date) AS max_date,
        1 AS snapshot_rank
    FROM 
        online_retail

    UNION ALL

    -- Recursive Member
    SELECT
        DATE_ADD(snapshot, INTERVAL 2 MONTH) AS snapshot,
        max_date,
        snapshot_rank + 1 AS snapshot_rank
    FROM 
        snapshots
    WHERE
        DATE_ADD(snapshot, INTERVAL 2 MONTH) <= max_date
),

online_retail_staged AS (
    SELECT
        o.country,
        o.description,
        o.quantity,
        o.invoice_date,
        o.first_purchase_date,
        o.unit_price,
        o.customer_id,
        o.invoice_number,
        o.order_value,
        s.snapshot,
        s.snapshot_rank
    FROM
        online_retail o
    LEFT JOIN
        snapshots s ON 
        o.invoice_date >= s.snapshot 
        AND o.invoice_date < DATE_ADD(s.snapshot, INTERVAL 2 MONTH)
),

discounts AS (
    SELECT
        customer_id,
        snapshot,
        SUM(order_value) AS discount,
        COUNT(invoice_number) AS number_of_discounts 
    FROM
        online_retail_staged
    WHERE
        description = 'Discount'
    GROUP BY
        snapshot,
        customer_id
),

returns_and_cancellations AS (
    SELECT
        customer_id,
        snapshot,
        SUM(quantity) *-1                   AS units_returned,
        SUM(order_value) *-1                AS total_returns_value,
        COUNT(DISTINCT invoice_number)      AS number_of_orders_returned
    FROM
        online_retail_staged
    WHERE
        quantity < 0 -- Negative quantities are returned / cancelled orders
    GROUP BY
        snapshot,
        customer_id
),

snapshot_behaviour AS (
    SELECT
        o.customer_id,
        o.snapshot, 
        DATE_DIFF('day', MIN(o.first_purchase_date), MAX(o.invoice_date)) AS customer_age,
        SUM(o.order_value) AS snapshot_spend,
        COUNT(DISTINCT o.invoice_number) AS snapshot_orders_placed,
        SUM(o.order_value) / COUNT(DISTINCT o.invoice_number) AS snapshot_avg_order_value,
        MAX(d.number_of_discounts) AS snapshot_number_of_discounts,
        SUM(d.discount) AS snapshot_discount_amount,
        MAX(r.units_returned) AS snapshot_units_returned,
        SUM(r.total_returns_value) AS snapshot_returns_amount,
        MAX(r.number_of_orders_returned) AS snapshot_returned_orders
    FROM
        online_retail_staged o
    LEFT JOIN
        discounts d USING(snapshot, customer_id)
    LEFT JOIN
        returns_and_cancellations r USING(snapshot, customer_id)
    WHERE
        description = UPPER(description) -- Filter to just products
    GROUP BY
        snapshot,
        customer_id
)

SELECT * FROM customer_behaviour
'''
).df()

df.head(5)

CatalogException: Catalog Error: Table with name customer_behaviour does not exist!
Did you mean "sqlite_schema"?

In [57]:
df = duckdb.sql(
'''
WITH RECURSIVE online_retail AS (
    SELECT
        b.Country                                                   AS country,
        b.Description                                               AS description,
        b.Quantity                                                  AS quantity,
        CAST(STRPTIME(b.InvoiceDate, '%m/%d/%Y %H:%M') AS DATE)     AS invoice_date,
        MIN(invoice_date) OVER(PARTITION BY CustomerID)             AS first_purchase_date,
        b.UnitPrice                                                 AS unit_price,
        CAST(b.CustomerID AS STRING)                                AS customer_id,
        b.InvoiceNo                                                 AS invoice_number,
        b.Quantity * b.UnitPrice                                    AS order_value
    FROM
        data b
    WHERE 
        customer_id IS NOT NULL
),

snapshots AS (
    -- Anchor member - first date in dataset: 1/10/2010
    SELECT
        MIN(invoice_date) AS snapshot,
        MAX(invoice_date) AS max_date
    FROM 
        online_retail

    UNION ALL

    -- Recursive Member
    SELECT
        DATE_ADD(snapshot, INTERVAL 2 MONTH) AS snapshot,
        max_date
    FROM 
        snapshots
    WHERE
        DATE_ADD(snapshot, INTERVAL 2 MONTH) <= max_date
),

customer_snapshots AS (
    SELECT 
        s.snapshot,
        c.customer_id
    FROM 
        snapshots s
    CROSS JOIN 
        (SELECT DISTINCT customer_id FROM online_retail) c
),

cleaned_transactions AS (
    SELECT
        customer_id,
        invoice_number,
        invoice_date,
        description,
        order_value,
        quantity,
        CASE WHEN description = 'Discount' THEN 1 ELSE 0 END AS is_discount,
        CASE WHEN description = 'Discount' THEN order_value ELSE 0 END AS discount_value,
        CASE WHEN quantity < 0 THEN 1 ELSE 0 END AS is_return,
        CASE WHEN quantity < 0 THEN order_value * -1 ELSE 0 END AS returns_value,
        CASE WHEN ASCII(description) = ASCII(UPPER(description)) THEN 1 ELSE 0 END AS is_product
    FROM
        online_retail
),

snapshot_behaviour AS (
    SELECT 
        s.snapshot, 
        s.customer_id,
        SUM(c.order_value)                                                      AS snapshot_order_value,
        COUNT(DISTINCT c.invoice_number)                                        AS snapshot_orders_placed,
        SUM(c.order_value) / COUNT(DISTINCT c.invoice_number)                   AS snapshot_avg_order_value,
        COUNT(DISTINCT CASE WHEN c.is_discount = 1 THEN c.invoice_number END)   AS snapshot_total_discounts,
        COUNT(DISTINCT CASE WHEN c.is_return = 1 THEN c.invoice_number END)     AS snapshot_orders_returned,
        COUNT(DISTINCT CASE WHEN c.is_product = 1 THEN c.description END)       AS snapshot_unique_products_ordered,
        SUM(c.discount_value)                                                   AS snapshot_discount_value,
        SUM(c.returns_value)                                                    AS snapshot_returns_value,
    FROM
        customer_snapshots s
    LEFT JOIN
        cleaned_transactions c 
        ON c.customer_id = s.customer_id
        AND c.invoice_date >= s.snapshot  
        AND c.invoice_date < DATE_ADD(s.snapshot, INTERVAL 2 MONTH) -- Looking at only the current snapshot
    GROUP BY
        s.snapshot,
        s.customer_id
),

customer_history AS (
    SELECT
        s.snapshot, 
        s.customer_id,
        SUM(c.order_value)                                                      AS total_order_value,
        COUNT(DISTINCT c.invoice_number)                                        AS total_orders_placed,
        SUM(c.order_value) / COUNT(DISTINCT c.invoice_number)                   AS historic_avg_order_value,
        COUNT(DISTINCT CASE WHEN c.is_discount = 1 THEN c.invoice_number END)   AS total_discounts,
        COUNT(DISTINCT CASE WHEN c.is_return = 1 THEN c.invoice_number END)     AS total_orders_returned,
        COUNT(DISTINCT CASE WHEN c.is_product = 1 THEN c.description END)       AS unique_products_ordered,
        SUM(c.discount_value)                                                   AS total_discount_value,
        SUM(c.returns_value)                                                    AS total_returns_value
    FROM
        customer_snapshots s
    LEFT JOIN
        cleaned_transactions c
        ON c.customer_id = s.customer_id
        AND c.invoice_date < s.snapshot -- Looking at behaviour in all previous snapshots
    GROUP BY
        s.snapshot,
        s.customer_id
)

SELECT
    COALESCE(a.customer_id, b.customer_id) AS customer_id,
    COALESCE(a.snapshot, b.snapshot) AS snapshot,

    a.total_order_value,
    a.total_orders_placed,
    a.historic_avg_order_value,
    a.total_discounts,
    a.total_orders_returned,
    a.unique_products_ordered,
    a.total_discount_value,
    a.total_returns_value,

    b.snapshot_order_value,
    b.snapshot_orders_placed,
    b.snapshot_avg_order_value,
    b.snapshot_total_discounts,
    b.snapshot_orders_returned,
    b.snapshot_unique_products_ordered,
    b.snapshot_discount_value,
    b.snapshot_returns_value,

-- Churn Flag: 1 if the NEXT snapshot period has ZERO or NULL orders
CASE 
    WHEN COALESCE(LEAD(snapshot_orders_placed, 1) OVER (
        PARTITION BY customer_id 
        ORDER BY snapshot
    ), 0) = 0 THEN 1 
    ELSE 0 
END AS customer_churned
FROM
    customer_history a 
LEFT JOIN
    snapshot_behaviour b USING(snapshot, customer_id)

QUALIFY LAG(snapshot_orders_placed, 1, 1) OVER (
    PARTITION BY customer_id
    ORDER BY snapshot
) > 0
'''
).df()

df

,customer_id,snapshot,total_order_value,total_orders_placed,historic_avg_order_value,total_discounts,total_orders_returned,unique_products_ordered,total_discount_value,total_returns_value,snapshot_order_value,snapshot_orders_placed,snapshot_avg_order_value,snapshot_total_discounts,snapshot_orders_returned,snapshot_unique_products_ordered,snapshot_discount_value,snapshot_returns_value,customer_churned
0,12353.0,2010-12-01,NaN,0,NaN,0,0,0,NaN,NaN,NaN,0,NaN,0,0,0,NaN,NaN,1
1,12353.0,2011-06-01,89.00,1,89.000000,0,0,4,0.0,0.0,NaN,0,NaN,0,0,0,NaN,NaN,1
2,12358.0,2010-12-01,NaN,0,NaN,0,0,0,NaN,NaN,NaN,0,NaN,0,0,0,NaN,NaN,1
3,12358.0,2011-08-01,484.86,1,484.860000,0,0,12,0.0,0.0,NaN,0,NaN,0,0,0,NaN,NaN,1
4,12407.0,2010-12-01,NaN,0,NaN,0,0,0,NaN,NaN,NaN,0,NaN,0,0,0,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14680,18235.0,2011-06-01,430.46,1,430.460000,0,0,29,0.0,0.0,NaN,0,NaN,0,0,0,NaN,NaN,0
14681,18235.0,2011-10-01,1796.48,3,598.826667,0,0,100,0.0,0.0,NaN,0,NaN,0,0,0,NaN,NaN,1
14682,18248.0,2010-12-01,NaN,0,NaN,0,0,0,NaN,NaN,NaN,0,NaN,0,0,0,NaN,NaN,1
14683,18248.0,2011-08-01,496.46,1,496.460000,0,0,28,0.0,0.0,286.56,3,95.52,0,2,19,0.0,21.0,1
